# Developing a PCMCI Trading Strategy — Field Notes

A practical playbook for going from *causal discovery* to a *tradable, honestly-evaluated* signal, distilled from building the `pcmci_stocks_demo` notebook. Read this before starting a new PCMCI strategy so you don't repeat the same mistakes.

**TL;DR of the whole journey**

| Stage | What changed | OOS result |
|---|---|---|
| 1. Raw returns | PCMCI+ on price log-returns, trade every link | Sharpe ≈ **-2.5** (mostly beta contamination) |
| 2. Confidence filter | Keep only strong `|MCI|` links / top-k conviction | *No rescue* — strong links fail OOS too |
| 3. Factor-neutralize | Strip market/sector factors (PCA) *before* discovery | Sharpe ≈ **-0.4** gross (flat; real links, but costs bite) |
| 4. Cut turnover | Slow the signal + no-trade band | Sharpe ≈ **+0.4** net of 5 bps |

The lesson isn't "we found alpha" (we didn't, robustly — see the honesty section). It's that **most of the loss was avoidable methodology error**, and fixing it in the right order is the actual skill.

## 1. What PCMCI actually gives you

PCMCI answers a stronger question than correlation: *does $X^i_{t-\tau}$ **drive** $X^j_t$ after removing the influence of everything else?* It runs in two stages:

1. **PC1 (condition selection)** — iteratively prunes each variable's candidate parent set. `pc_alpha` is the pruning knob; treat it as a **regularizer, not a p-value**.
2. **MCI test** — for each surviving link, tests conditional independence given the parents of **both** cause and effect. Conditioning on both is what makes it robust to autocorrelation (the usual source of spurious links in time series).

| Variant | Adds | Use when |
|---|---|---|
| PCMCI | lagged links only | clean lagged story |
| **PCMCI+** | + contemporaneous (same-bar) | intraday/daily co-movement matters |
| LPCMCI | + tolerates latent confounders | you can't fully factor-neutralize |

**Output** is a `graph` array (`'-->'` directed, `'o-o'`/`'x-x'` unoriented) and a `val_matrix` of MCI partial-correlation strengths. A lagged `A --> B (lag 1)` means: today's A carries information about tomorrow's B not already in B's own past — **the raw material of a lead-lag signal.**

## 2. The assumptions — and exactly how markets break them

PCMCI is only valid under four assumptions. Markets violate two of them hard, and that is the source of almost every bad result:

| Assumption | Meaning | Market reality | Fix |
|---|---|---|---|
| **Causal sufficiency** | no hidden common driver | **VIOLATED** — market & sector factors drive everything | **Factor-neutralize first** (PCA/market-model residuals) or use LPCMCI |
| **Causal stationarity** | mechanism constant in time | **VIOLATED** — regimes, vol clustering | rolling windows, RPCMCI, shorter fit windows |
| Markov | present screens past | approx OK on returns | use returns, not prices |
| Faithfulness | no exact cancellation | usually OK | — |

> **The #1 mistake:** running PCMCI on raw returns. When two stocks co-move because the *market* moved, PCMCI can label that shared beta as a lead-lag "cause." You then trade a pile of correlated beta bets that get whipsawed. **Strip the factors before discovery.**

## 3. The end-to-end recipe

```
prices ──► log returns ──► FACTOR-NEUTRALIZE (PCA, train-fit) ──► standardize
                                    │
                                    ▼
            PCMCI+ on TRAIN residuals only  ──►  lead-lag graph
                                    │
                                    ▼
        per-stock OLS: next resid return ~ leaders' lagged residuals (fit on TRAIN)
                                    │
                                    ▼
     predict every day ──► cross-sectional demean ──► dollar-neutral weights
                                    │
                                    ▼
        SLOW the signal (holding window) + NO-TRADE BAND  ──►  low turnover
                                    │
                                    ▼
         backtest OOS on REAL raw returns, NET of costs + turnover
```

Each arrow corresponds to a lesson learned the hard way. The order matters: factor-neutralization (biggest lever) before turnover control before parameter tuning.

## 4. The non-negotiable honesty rules

Break any of these and your backtest will look brilliant and trade terribly:

1. **Discover on train only.** Run PCMCI, fit OLS betas, *and* fit PCA loadings on the training slice. Never let test data touch discovery or fitting.
2. **No lookahead in timing.** `pred[t]` may use only `R[t-τ]` for `τ ≥ 1` (known at close of `t-1`), so weights are held *into* day `t`. Signal smoothing/bands must be past-only too.
3. **Evaluate net of costs and turnover.** A flat gross curve with 1.3×/day turnover is a *losing* strategy at 5 bps. Always report net Sharpe and average turnover.
4. **Don't tune on the test set.** Sweeping `(hold, band)` or `|MCI|` thresholds and picking the best on the test slice is p-hacking. Use a **validation split**, or trust only the *direction/monotonicity* of a sweep, not its peak number.
5. **Have a benchmark.** Compare against 1-day cross-sectional reversal (or similar). If you can't beat a one-line baseline, you have nothing.

## 5. Reference implementation (skeleton)

The cell below is a compact, copy-paste skeleton of the full pipeline. It assumes `R` is a `(T, N)` array of raw daily log returns and `stock_names` is the column list. It is deliberately minimal — the point is the *structure* and the *no-lookahead discipline*, not production robustness.

In [ ]:
import numpy as np
import tigramite.data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

ANN = 252

def factor_neutralize(R, split, k_fac=5):
    """Strip top-k statistical factors using TRAIN-fit PCA loadings. No lookahead."""
    mu, sd = R[:split].mean(0), R[:split].std(0)
    Z = (R - mu) / sd                       # standardize with TRAIN stats
    _, _, Vt = np.linalg.svd(Z[:split] - Z[:split].mean(0), full_matrices=False)
    load = Vt[:k_fac]                        # (k, N) factor directions from TRAIN
    return Z - (Z @ load.T) @ load           # idiosyncratic residuals

def discover_leaders(resid, split, stock_names, tau_max=2, pc_alpha=0.05):
    """PCMCI+ on TRAIN residuals -> {effect: [(cause, lag), ...]}"""
    N = resid.shape[1]
    g = PCMCI(dataframe=pp.DataFrame(resid[:split], var_names=stock_names),
              cond_ind_test=ParCorr(significance='analytic'), verbosity=0
              ).run_pcmciplus(tau_min=0, tau_max=tau_max, pc_alpha=pc_alpha)['graph']
    lead = {j: [] for j in range(N)}
    for i in range(N):
        for j in range(N):
            for tau in range(1, tau_max + 1):
                if g[i, j, tau] == '-->' and i != j:
                    lead[j].append((i, tau))
    return lead

def fit_predictor(resid, lead, split, tau_max=2):
    """Per-stock OLS on TRAIN residuals; predict next-day resid for all days."""
    T, N = resid.shape
    P = np.full((T, N), np.nan)
    rows = np.arange(tau_max, T)
    for j in range(N):
        L = lead[j]
        if not L:
            continue
        design = np.column_stack([np.ones(len(rows))] + [resid[rows - t, i] for (i, t) in L])
        y, tr = resid[rows, j], rows < split
        b, *_ = np.linalg.lstsq(design[tr], y[tr], rcond=None)
        P[rows, j] = design @ b
    return P

def slow_signal(pred, hold):
    if hold <= 1:
        return pred
    out = np.full_like(pred, np.nan)
    for t in range(pred.shape[0]):
        out[t] = np.nanmean(pred[max(0, t - hold + 1):t + 1], axis=0)
    return out

def backtest(pred, R, start, band=0.0, cost_bps=0.0):
    """Dollar-neutral book, no-trade band, PnL on REAL raw returns, net of costs."""
    T, N = R.shape
    w_prev = np.zeros(N); rets, turn = [], []
    for t in range(start, T):
        p = pred[t]; ok = np.isfinite(p)
        w_tgt = np.zeros(N)
        if ok.sum() >= 4:
            s = np.zeros(N); s[ok] = p[ok] - p[ok].mean()
            g = np.abs(s).sum()
            if g > 0:
                w_tgt = s / g
        w = w_prev.copy()
        trade = np.abs(w_tgt - w_prev) > band
        w[trade] = w_tgt[trade]
        g2 = np.abs(w).sum()
        if g2 > 0:
            w = w / g2
        tc = cost_bps * 1e-4 * np.abs(w - w_prev).sum()
        rets.append((w * R[t]).sum() - tc)
        turn.append(np.abs(w - w_prev).sum())
        w_prev = w
    return np.array(rets), np.array(turn)

def stats(rets):
    mu, sg = rets.mean(), rets.std()
    return (mu / sg * np.sqrt(ANN) if sg > 0 else 0.0,
            (rets > 0).mean(), np.prod(1 + rets) - 1)

# Usage:
# split = int(0.70 * len(R))
# resid = factor_neutralize(R, split, k_fac=5)
# lead  = discover_leaders(resid, split, stock_names)
# pred  = slow_signal(fit_predictor(resid, lead, split), hold=10)
# r, tn = backtest(pred, R, split, band=0.02, cost_bps=5.0)
# print(stats(r), 'avg turnover', tn.mean())

## 6. Parameter guide

| Knob | What it does | Sensible start | Notes |
|---|---|---|---|
| `pc_alpha` | PC1 pruning aggressiveness | `0.05` | permissive on daily returns (they're nearly unpredictable); tighten if too many links |
| `tau_max` | max lag searched | `2` (days) | daily lead-lag decays fast; bigger just adds noise |
| `tau_min` | `0` includes contemporaneous | `0` for PCMCI+ | same-bar direction is hard to identify — don't over-trust it |
| `k_fac` (PCA) | # factors removed | `5` | factor 1 ≈ market, next few ≈ sectors; ~50–60% of variance |
| `hold` (smoothing) | signal averaging window | `5–10` days | the single biggest turnover lever |
| `band` (no-trade) | min weight move to trade | `0.01–0.02` | leaves small drifts alone |
| `cost_bps` | per-side cost assumption | `5` (realistic-ish) | **always** report a net number |
| `fdr_method` | multiple-testing control | `'fdr_bh'` | with 50 names you run thousands of tests |

## 7. Pitfalls checklist

Run through this before believing any PCMCI backtest:

- [ ] Ran on **returns**, not prices (stationarity).
- [ ] **Factor-neutralized before discovery** (causal sufficiency) — this alone moved Sharpe ~+2.
- [ ] PCMCI, OLS betas, **and** PCA loadings all fit on **train only**.
- [ ] Weights held **into** the day they earn (no lookahead); smoothing is past-only.
- [ ] Reported **net of costs** with **turnover** shown.
- [ ] Did **not** pick sweep parameters on the test set (used validation or trusted only monotonic trend).
- [ ] Compared against a **baseline** (e.g. 1-day reversal).
- [ ] Sanity-checked link count: too many links at `pc_alpha=0.05` on raw returns is a red flag for factor contamination.
- [ ] Remembered the result is **illustrative** on ~2y daily / 50 names — a weak, fast-decaying edge, not investable.

## 8. Where the edge actually lives (roadmap)

Ranked by expected impact, once the pipeline above is correct:

1. **Higher frequency.** Daily single-name lead-lag is genuinely weak and decays fast; intraday (hourly/5-min) lead-lag is the stronger, more-published edge (Bennett–Cucuringu).
2. **Lower turnover / longer holding.** Costs — not signal sign — are usually what kill a factor-neutral signal. This is the cheapest win after neutralization.
3. **Stability filtering.** Bagged-PCMCI+ or rolling re-estimation; keep only links stable across resamples/windows. Reduces noise trades.
4. **Network clustering.** Trade cluster-level lead-lag signals (Bennett–Cucuringu–Reinert) rather than fragile single pairs.
5. **Dynamic lags.** Static precomputed graphs underperform pair-specific, time-varying lags (DeltaLag, arXiv:2511.00390).
6. **Economic priors.** Keep only economically plausible links (supplier–customer, same sector). Mainly *shrinks losing trades* rather than adding gains (Kim et al.).
7. **Shrinkage / rank signals.** With tiny samples the OLS betas are noisy — ridge them, or trade sign/rank instead of fitted magnitude.
8. **Nonlinear CI tests.** `GPDC`/`CMIknn` if you suspect nonlinear lead-lag — slower, and won't fix a fundamentally weak edge.

## 9. Resources

- **Tigramite** (PCMCI implementation) — https://github.com/jakobrunge/tigramite
- **PCMCI paper** — Runge et al., *Detecting and quantifying causal associations in large nonlinear time series datasets*, Science Advances 2019 — https://www.science.org/doi/10.1126/sciadv.aau4996
- Bennett, Cucuringu & Reinert (2022), *Lead-lag detection and network clustering for the US equity market* — arXiv:2201.08283
- Oliveira, Lu, Lin, Cucuringu & Fujita (2024), *Causality-Inspired Models for Financial Time Series Forecasting* — arXiv:2408.09960
- Zhou et al. (2025), *DeltaLag* — arXiv:2511.00390
- Kim et al. (2026), *LLM as a Risk Manager* — arXiv:2602.07048
- Xu, Cheng & Lee (2025), *A Causal Perspective of Stock Prediction* — arXiv:2503.20987

*Companion working notebook: `pcmci_stocks_demo.ipynb` (Parts 1–6 implement everything referenced here).*